# Unit10_Stepwise_Regression | 副標題：逐步回歸與特徵選擇

本 Notebook 使用模擬的化學催化反應數據，示範 **逐步回歸 (Stepwise Regression)** 的三種策略：**前向選擇 (Forward Selection)**、**後向消除 (Backward Elimination)** 與 **雙向逐步 (Bidirectional Stepwise)**，並與 Lasso、OLS 進行比較。

## 目標
- 理解逐步回歸作為「包裹式 (Wrapper)」特徵選擇方法的原理
- 使用 `statsmodels` 手動實作以 AIC 與 p-value 為準則的逐步選擇
- 使用 sklearn `SequentialFeatureSelector` 進行交叉驗證式特徵選擇
- 比較逐步回歸、Lasso、OLS 選到的特徵與預測性能
- 視覺化特徵選擇路徑（AIC 隨步驟變化）
- 評估模型穩定性並討論逐步回歸的推論偏誤限制

## 數據設計
本案例採用 **8 個特徵** 設計（4 重要 + 4 冗餘），其中 4 個重要特徵的真實係數與 Unit10_Linear_Regression 單元一致，便於跨單元對照：
- **4 個重要特徵**：溫度、壓力、催化劑濃度、反應時間（真實影響產率）
- **4 個冗餘特徵**：攪拌速度、進料速率、pH、溶劑比（不影響產率）

**目標**：驗證逐步回歸能否自動識別並保留 4 個重要特徵、剔除 4 個冗餘特徵。

---
### 0. 環境設定

In [1]:
from pathlib import Path
import os

# ========================================
# 路徑設定 (兼容 Colab 與 Local)
# ========================================
UNIT_OUTPUT_DIR = 'P3_Unit10_Stepwise_Regression'

try:
  from google.colab import drive
  IN_COLAB = True
  print("✓ 偵測到 Colab 環境，準備掛載 Google Drive...")
  drive.mount('/content/drive', force_remount=True)
except ImportError:
  IN_COLAB = False
  print("✓ 偵測到 Local 環境")

try:
  shortcut_path = '/content/ChemE-3590'
  os.remove(shortcut_path)
except FileNotFoundError:
  pass

if IN_COLAB:
  source_path = Path('/content/drive/My Drive/Colab Notebooks/ChemE-3590')
  os.symlink(source_path, shortcut_path)
  shortcut_path = Path(shortcut_path)
  if source_path.exists():
    NOTEBOOK_DIR = shortcut_path / 'Part_3' / 'Unit10'
    OUTPUT_DIR = NOTEBOOK_DIR / 'outputs' / UNIT_OUTPUT_DIR
    MODEL_DIR = OUTPUT_DIR / 'models'
    FIG_DIR = OUTPUT_DIR / 'figs'
  else:
    print(f"⚠️ 找不到路徑雲端ChemE-3590路徑，請確認自己的雲端資料夾是否正確")
else:
  NOTEBOOK_DIR = Path.cwd()
  OUTPUT_DIR = NOTEBOOK_DIR / 'outputs' / UNIT_OUTPUT_DIR
  MODEL_DIR = OUTPUT_DIR / 'models'
  FIG_DIR = OUTPUT_DIR / 'figs'

NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n✓ Notebook工作目錄: {NOTEBOOK_DIR}")
print(f"✓ 結果輸出目錄: {OUTPUT_DIR}")
print(f"✓ 模型輸出目錄: {MODEL_DIR}")
print(f"✓ 圖檔輸出目錄: {FIG_DIR}")

✓ 偵測到 Local 環境

✓ Notebook工作目錄: D:\MyGit\ChemE-3590\Part_3\Unit10
✓ 結果輸出目錄: D:\MyGit\ChemE-3590\Part_3\Unit10\outputs\P3_Unit10_Stepwise_Regression
✓ 模型輸出目錄: D:\MyGit\ChemE-3590\Part_3\Unit10\outputs\P3_Unit10_Stepwise_Regression\models
✓ 圖檔輸出目錄: D:\MyGit\ChemE-3590\Part_3\Unit10\outputs\P3_Unit10_Stepwise_Regression\figs


---
### 1. 載入套件

In [2]:
# 基礎套件
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json, joblib
import warnings
warnings.filterwarnings('ignore')

# 機器學習套件
from sklearn.linear_model import LinearRegression, Lasso, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# 統計模型 (逐步回歸的 AIC / p-value 來源)
import statsmodels.api as sm

# 設定隨機種子
np.random.seed(42)

# 繪圖設定
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

print("✓ 套件載入完成")

✓ 套件載入完成


---
### 2. 數據生成（模擬化學催化反應）

設計 8 個特徵，其中僅前 4 個真正影響產率，後 4 個為冗餘（真實係數為 0），用來測試逐步回歸的特徵選擇能力。

In [3]:
# 設定樣本數
n_samples = 500

# === 生成 8 個特徵 ===
# 重要特徵（真正影響產率）
temperature = np.random.uniform(80, 120, n_samples)   # Temperature (deg C)
pressure = np.random.uniform(1, 5, n_samples)         # Pressure (bar)
catalyst = np.random.uniform(0.5, 2.5, n_samples)     # Catalyst (wt%)
time = np.random.uniform(30, 180, n_samples)          # Time (min)

# 冗餘特徵（不影響產率）
stirring_speed = np.random.uniform(100, 500, n_samples)  # Stirring Speed (rpm)
feed_rate = np.random.uniform(10, 50, n_samples)         # Feed Rate (L/h)
ph = np.random.uniform(6.0, 8.0, n_samples)              # pH
solvent_ratio = np.random.uniform(0.1, 0.5, n_samples)   # Solvent Ratio

# === 真實模型係數（僅前 4 個特徵有貢獻）===
true_intercept = 20.0
true_coef_temp = 0.5
true_coef_pressure = 3.0
true_coef_catalyst = 5.0
true_coef_time = 0.05

true_yield = (true_intercept +
              true_coef_temp * temperature +
              true_coef_pressure * pressure +
              true_coef_catalyst * catalyst +
              true_coef_time * time)

# 加入雜訊
noise = np.random.normal(0, 3, n_samples)
yield_percent = true_yield + noise

df = pd.DataFrame({
    'Temperature_C': temperature,
    'Pressure_bar': pressure,
    'Catalyst_wt%': catalyst,
    'Time_min': time,
    'Stirring_Speed_rpm': stirring_speed,
    'Feed_Rate_Lh': feed_rate,
    'pH': ph,
    'Solvent_Ratio': solvent_ratio,
    'Yield_%': yield_percent
})

print("✓ 數據生成完成")
print(f"數據集大小: {df.shape}")
print(f"特徵數量: {df.shape[1] - 1} (其中 4 個重要，4 個冗餘)")
print("\n前5筆數據:")
print(df.head())
print("\n數據統計摘要:")
print(df.describe().round(3))

✓ 數據生成完成
數據集大小: (500, 9)
特徵數量: 8 (其中 4 個重要，4 個冗餘)

前5筆數據:
   Temperature_C  Pressure_bar  Catalyst_wt%    Time_min  Stirring_Speed_rpm  \
0      94.981605      3.792647      0.870266  107.862268          204.682273   
1     118.028572      3.144385      1.583802  101.877282          198.791520   
2     109.279758      2.238110      2.245892   33.846310          462.501832   
3     103.946339      4.255180      1.964450   81.187174          199.818480   
4      86.240746      3.738925      2.113122   87.029343          208.779890   

   Feed_Rate_Lh        pH  Solvent_Ratio     Yield_%  
0     43.100757  7.345406       0.214608   92.599108  
1     40.581112  7.593363       0.421056  102.399870  
2     32.941158  6.500936       0.498895   92.456474  
3     48.241886  7.249748       0.112010   99.988030  
4     18.018981  7.143492       0.458946   87.876955  

數據統計摘要:
       Temperature_C  Pressure_bar  Catalyst_wt%  Time_min  \
count        500.000       500.000       500.000   500.000  

---
### 3. 數據探索：相關係數分析

檢視各特徵與產率的相關性。重要特徵應與 `Yield_%` 呈現明顯相關，冗餘特徵則接近 0。

In [4]:
# 計算相關係數矩陣
corr = df.corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'correlation_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

# 各特徵與目標的相關係數
target_corr = corr['Yield_%'].drop('Yield_%').sort_values(key=abs, ascending=False)
print("各特徵與 Yield_% 的相關係數 (依絕對值排序):")
print(target_corr.round(4))

各特徵與 Yield_% 的相關係數 (依絕對值排序):
Temperature_C         0.7109
Pressure_bar          0.4082
Catalyst_wt%          0.3916
Time_min              0.2459
pH                   -0.0706
Stirring_Speed_rpm    0.0344
Solvent_Ratio         0.0344
Feed_Rate_Lh          0.0280
Name: Yield_%, dtype: float64


---
### 4. 資料準備：切分與標準化

- 切分訓練集 (80%) 與測試集 (20%)
- 使用 `StandardScaler` 標準化特徵（逐步回歸中標準化不改變 AIC/p-value 的選擇結果，但便於與 Lasso 的係數比較）

In [5]:
# 分離特徵與目標
feature_names = [c for c in df.columns if c != 'Yield_%']
X = df[feature_names]
y = df['Yield_%']

# 切分數據集（80% 訓練，20% 測試）
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# 標準化（以訓練集統計量為準）
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), columns=feature_names, index=X_train.index)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), columns=feature_names, index=X_test.index)

print(f"✓ 訓練集: {X_train_scaled.shape}, 測試集: {X_test_scaled.shape}")
print(f"✓ 標準化後訓練集均值 (應接近 0):\n{X_train_scaled.mean().round(4)}")

✓ 訓練集: (400, 8), 測試集: (100, 8)
✓ 標準化後訓練集均值 (應接近 0):
Temperature_C        -0.0
Pressure_bar          0.0
Catalyst_wt%          0.0
Time_min             -0.0
Stirring_Speed_rpm    0.0
Feed_Rate_Lh          0.0
pH                    0.0
Solvent_Ratio        -0.0
dtype: float64


---
### 5. 定義逐步回歸工具函數

以 `statsmodels` 的 OLS 為基礎，實作：
- **前向選擇 (AIC)**：從空模型開始，每步加入使 AIC 下降最多的特徵，直到無法再降。
- **後向消除 (AIC)**：從全模型開始，每步移除使 AIC 下降最多的特徵。
- **雙向逐步 (p-value)**：交替進行「加入顯著特徵」與「移除不顯著特徵」。

> AIC（赤池資訊準則） $= 2k - 2\ln(\hat{L})$ ，同時衡量模型配適度與複雜度，數值越小越好。

In [6]:
def forward_selection_aic(X, y, verbose=True):
    """前向選擇：以 AIC 為準則逐步加入特徵。"""
    remaining = list(X.columns)
    selected = []
    const = np.ones((len(y), 1))
    current_aic = sm.OLS(y, const).fit().aic  # 空模型 (僅截距)
    history = [('null', current_aic, [])]
    improved = True
    while remaining and improved:
        improved = False
        best_aic, best_feat = current_aic, None
        for feat in remaining:
            model = sm.OLS(y, sm.add_constant(X[selected + [feat]])).fit()
            if model.aic < best_aic:
                best_aic, best_feat = model.aic, feat
        if best_feat is not None:
            selected.append(best_feat)
            remaining.remove(best_feat)
            current_aic = best_aic
            history.append((f'+{best_feat}', current_aic, list(selected)))
            improved = True
            if verbose:
                print(f"  加入 {best_feat:20s} -> AIC = {current_aic:.2f}")
    return selected, history


def backward_elimination_aic(X, y, verbose=True):
    """後向消除：以 AIC 為準則逐步移除特徵。"""
    selected = list(X.columns)
    current_aic = sm.OLS(y, sm.add_constant(X[selected])).fit().aic
    history = [('full', current_aic, list(selected))]
    improved = True
    while len(selected) > 1 and improved:
        improved = False
        best_aic, worst_feat = current_aic, None
        for feat in selected:
            cols = [c for c in selected if c != feat]
            model = sm.OLS(y, sm.add_constant(X[cols])).fit()
            if model.aic < best_aic:
                best_aic, worst_feat = model.aic, feat
        if worst_feat is not None:
            selected.remove(worst_feat)
            current_aic = best_aic
            history.append((f'-{worst_feat}', current_aic, list(selected)))
            improved = True
            if verbose:
                print(f"  移除 {worst_feat:20s} -> AIC = {current_aic:.2f}")
    return selected, history


def stepwise_pvalue(X, y, threshold_in=0.05, threshold_out=0.05, verbose=True):
    """雙向逐步：以 p-value 為準則交替加入/移除特徵。"""
    included = []
    history = []
    while True:
        changed = False
        # 前向步驟：加入最顯著且 p < threshold_in 的特徵
        excluded = [c for c in X.columns if c not in included]
        new_pval = pd.Series(index=excluded, dtype=float)
        for feat in excluded:
            model = sm.OLS(y, sm.add_constant(X[included + [feat]])).fit()
            new_pval[feat] = model.pvalues[feat]
        if not new_pval.empty and new_pval.min() < threshold_in:
            best_feat = new_pval.idxmin()
            included.append(best_feat)
            history.append((f'+{best_feat}', float(new_pval.min())))
            changed = True
            if verbose:
                print(f"  加入 {best_feat:20s} (p = {new_pval.min():.4g})")
        # 後向步驟：移除最不顯著且 p > threshold_out 的特徵
        if included:
            model = sm.OLS(y, sm.add_constant(X[included])).fit()
            pvals = model.pvalues.drop('const')
            if pvals.max() > threshold_out:
                worst_feat = pvals.idxmax()
                included.remove(worst_feat)
                history.append((f'-{worst_feat}', float(pvals.max())))
                changed = True
                if verbose:
                    print(f"  移除 {worst_feat:20s} (p = {pvals.max():.4g})")
        if not changed:
            break
    return included, history

print("✓ 逐步回歸工具函數已定義")

✓ 逐步回歸工具函數已定義


---
### 6. 前向選擇 (Forward Selection, AIC)

從空模型開始，逐步加入最能降低 AIC 的特徵。

In [7]:
print("=== 前向選擇 (AIC) ===")
fwd_selected, fwd_history = forward_selection_aic(X_train_scaled, y_train)
print(f"\n✓ 前向選擇結果 ({len(fwd_selected)} 個特徵):")
print(f"  {fwd_selected}")

=== 前向選擇 (AIC) ===
  加入 Temperature_C        -> AIC = 2568.44
  加入 Pressure_bar         -> AIC = 2412.41
  加入 Catalyst_wt%         -> AIC = 2227.62
  加入 Time_min             -> AIC = 2070.40
  加入 pH                   -> AIC = 2069.69

✓ 前向選擇結果 (5 個特徵):
  ['Temperature_C', 'Pressure_bar', 'Catalyst_wt%', 'Time_min', 'pH']


---
### 7. 後向消除 (Backward Elimination, AIC)

從全特徵模型開始，逐步移除移除後最能降低 AIC 的特徵。

In [8]:
print("=== 後向消除 (AIC) ===")
bwd_selected, bwd_history = backward_elimination_aic(X_train_scaled, y_train)
print(f"\n✓ 後向消除結果 ({len(bwd_selected)} 個特徵):")
print(f"  {bwd_selected}")

=== 後向消除 (AIC) ===
  移除 Stirring_Speed_rpm   -> AIC = 2071.01


  移除 Solvent_Ratio        -> AIC = 2070.17


  移除 Feed_Rate_Lh         -> AIC = 2069.69

✓ 後向消除結果 (5 個特徵):
  ['Temperature_C', 'Pressure_bar', 'Catalyst_wt%', 'Time_min', 'pH']


---
### 8. 雙向逐步 (Bidirectional Stepwise, p-value)

交替進行前向加入 (p < 0.05) 與後向移除 (p > 0.05)，兼顧兩種方向的調整。

In [9]:
print("=== 雙向逐步 (p-value, 門檻 0.05) ===")
step_selected, step_history = stepwise_pvalue(X_train_scaled, y_train)
print(f"\n✓ 雙向逐步結果 ({len(step_selected)} 個特徵):")
print(f"  {step_selected}")

# 三種策略選到的特徵彙整
important = ['Temperature_C', 'Pressure_bar', 'Catalyst_wt%', 'Time_min']
print("\n--- 三種逐步策略比較 ---")
for name, sel in [('Forward (AIC)', fwd_selected),
                  ('Backward (AIC)', bwd_selected),
                  ('Stepwise (p)', step_selected)]:
    hit = sorted(set(sel) & set(important))
    extra = sorted(set(sel) - set(important))
    print(f"{name:16s}: {len(sel)} 特徵 | 命中重要 {len(hit)}/4 | 多餘 {extra}")

=== 雙向逐步 (p-value, 門檻 0.05) ===
  加入 Temperature_C        (p = 2.909e-63)
  加入 Pressure_bar         (p = 6.086e-36)
  加入 Catalyst_wt%         (p = 4.57e-42)
  加入 Time_min             (p = 5.036e-36)



✓ 雙向逐步結果 (4 個特徵):
  ['Temperature_C', 'Pressure_bar', 'Catalyst_wt%', 'Time_min']

--- 三種逐步策略比較 ---
Forward (AIC)   : 5 特徵 | 命中重要 4/4 | 多餘 ['pH']
Backward (AIC)  : 5 特徵 | 命中重要 4/4 | 多餘 ['pH']
Stepwise (p)    : 4 特徵 | 命中重要 4/4 | 多餘 []


---
### 9. 特徵選擇路徑視覺化 (AIC)

繪製前向選擇與後向消除過程中 AIC 隨步驟的變化。

In [10]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 前向選擇路徑
fwd_labels = [h[0] for h in fwd_history]
fwd_aics = [h[1] for h in fwd_history]
axes[0].plot(range(len(fwd_aics)), fwd_aics, 'o-', color='tab:blue', linewidth=2)
axes[0].set_xticks(range(len(fwd_labels)))
axes[0].set_xticklabels(fwd_labels, rotation=45, ha='right')
axes[0].set_ylabel('AIC')
axes[0].set_title('Forward Selection Path (AIC)', fontweight='bold')
axes[0].grid(True, alpha=0.3)

# 後向消除路徑
bwd_labels = [h[0] for h in bwd_history]
bwd_aics = [h[1] for h in bwd_history]
axes[1].plot(range(len(bwd_aics)), bwd_aics, 's-', color='tab:red', linewidth=2)
axes[1].set_xticks(range(len(bwd_labels)))
axes[1].set_xticklabels(bwd_labels, rotation=45, ha='right')
axes[1].set_ylabel('AIC')
axes[1].set_title('Backward Elimination Path (AIC)', fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / 'stepwise_aic_path.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"前向最終 AIC: {fwd_aics[-1]:.2f} | 後向最終 AIC: {bwd_aics[-1]:.2f}")

前向最終 AIC: 2069.69 | 後向最終 AIC: 2069.69


---
### 10. sklearn SequentialFeatureSelector 對照

sklearn 提供以「交叉驗證評分」為準則的序列特徵選擇，與以 AIC/p-value 為準則的統計式逐步回歸互補。這裡以 5 折 CV 的 $R^2$ 為準則，固定選取 4 個特徵。

In [11]:
base = LinearRegression()

sfs_fwd = SequentialFeatureSelector(
    base, n_features_to_select=4, direction='forward', scoring='r2', cv=5)
sfs_fwd.fit(X_train_scaled, y_train)
sfs_fwd_feats = list(X_train_scaled.columns[sfs_fwd.get_support()])

sfs_bwd = SequentialFeatureSelector(
    base, n_features_to_select=4, direction='backward', scoring='r2', cv=5)
sfs_bwd.fit(X_train_scaled, y_train)
sfs_bwd_feats = list(X_train_scaled.columns[sfs_bwd.get_support()])

print("sklearn SFS (forward, 4 特徵):", sfs_fwd_feats)
print("sklearn SFS (backward, 4 特徵):", sfs_bwd_feats)
print("\n命中重要特徵:",
      f"forward {len(set(sfs_fwd_feats) & set(important))}/4,",
      f"backward {len(set(sfs_bwd_feats) & set(important))}/4")

sklearn SFS (forward, 4 特徵): ['Temperature_C', 'Pressure_bar', 'Catalyst_wt%', 'Time_min']
sklearn SFS (backward, 4 特徵): ['Temperature_C', 'Pressure_bar', 'Catalyst_wt%', 'Time_min']

命中重要特徵: forward 4/4, backward 4/4


---
### 11. 模型比較：Stepwise vs Lasso vs OLS

以逐步回歸（前向 AIC）選出的特徵訓練精簡 OLS，並與 LassoCV、全特徵 OLS 比較測試集性能。

In [12]:
def evaluate(model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    pred_te = model.predict(X_te)
    return {
        'R2_test': r2_score(y_te, pred_te),
        'RMSE_test': np.sqrt(mean_squared_error(y_te, pred_te)),
        'MAE_test': mean_absolute_error(y_te, pred_te),
    }

# (1) Stepwise 精簡 OLS (使用前向選擇特徵)
step_feats = fwd_selected
m_step = LinearRegression()
r_step = evaluate(m_step, X_train_scaled[step_feats], X_test_scaled[step_feats], y_train, y_test)

# (2) 全特徵 OLS
m_ols = LinearRegression()
r_ols = evaluate(m_ols, X_train_scaled, X_test_scaled, y_train, y_test)

# (3) LassoCV
alphas = np.logspace(-3, 1, 50)
m_lasso = LassoCV(alphas=alphas, cv=5, random_state=42, max_iter=10000)
r_lasso = evaluate(m_lasso, X_train_scaled, X_test_scaled, y_train, y_test)
lasso_nonzero = [f for f, c in zip(feature_names, m_lasso.coef_) if abs(c) > 1e-6]

compare = pd.DataFrame({
    'Stepwise_OLS': r_step,
    'Full_OLS': r_ols,
    'LassoCV': r_lasso,
}).T
compare['n_features'] = [len(step_feats), len(feature_names), len(lasso_nonzero)]
print("模型性能比較 (測試集):")
print(compare.round(4))
print(f"\nStepwise 選用特徵 ({len(step_feats)}): {step_feats}")
print(f"Lasso 非零特徵 ({len(lasso_nonzero)}): {lasso_nonzero}")
print(f"LassoCV 最佳 alpha: {m_lasso.alpha_:.4f}")

模型性能比較 (測試集):
              R2_test  RMSE_test  MAE_test  n_features
Stepwise_OLS   0.8703     2.9760    2.3400           5
Full_OLS       0.8711     2.9667    2.3353           8
LassoCV        0.8703     2.9761    2.3480           7

Stepwise 選用特徵 (5): ['Temperature_C', 'Pressure_bar', 'Catalyst_wt%', 'Time_min', 'pH']
Lasso 非零特徵 (7): ['Temperature_C', 'Pressure_bar', 'Catalyst_wt%', 'Time_min', 'Feed_Rate_Lh', 'pH', 'Solvent_Ratio']
LassoCV 最佳 alpha: 0.0429


---
### 11.1 特徵選擇對照視覺化

以熱圖呈現各方法是否選用每個特徵（1=選用，0=剔除），直觀比較不同方法的特徵選擇結果。

In [13]:
methods = {
    'Forward (AIC)': fwd_selected,
    'Backward (AIC)': bwd_selected,
    'Stepwise (p)': step_selected,
    'SFS forward': sfs_fwd_feats,
    'SFS backward': sfs_bwd_feats,
    'LassoCV': lasso_nonzero,
}
sel_matrix = pd.DataFrame(
    {m: [1 if f in feats else 0 for f in feature_names] for m, feats in methods.items()},
    index=feature_names).T

fig, ax = plt.subplots(figsize=(11, 4.5))
sns.heatmap(sel_matrix, annot=True, cmap='Greens', cbar=False,
            linewidths=1, linecolor='white', ax=ax)
ax.set_title('Feature Selection by Method (1 = selected)', fontweight='bold')
ax.set_xlabel('Features')
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.savefig(FIG_DIR / 'feature_selection_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

# 重要 vs 冗餘特徵的選中次數
print("各特徵被 6 種方法選中的次數:")
print(sel_matrix.sum().sort_values(ascending=False))

各特徵被 6 種方法選中的次數:
Temperature_C         6
Pressure_bar          6
Catalyst_wt%          6
Time_min              6
pH                    3
Feed_Rate_Lh          1
Solvent_Ratio         1
Stirring_Speed_rpm    0
dtype: int64


---
### 12. 預測結果評估 (Parity Plot)

以逐步回歸精簡模型在測試集上的預測值對實際值作圖，越接近對角線代表預測越準確。

In [14]:
m_step.fit(X_train_scaled[step_feats], y_train)
pred_test = m_step.predict(X_test_scaled[step_feats])

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_test, pred_test, alpha=0.6, edgecolor='k', linewidth=0.3, color='tab:blue')
lims = [min(y_test.min(), pred_test.min()), max(y_test.max(), pred_test.max())]
ax.plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
ax.set_xlabel('Actual Yield (%)')
ax.set_ylabel('Predicted Yield (%)')
ax.set_title(f'Stepwise OLS Parity Plot (R2 = {r_step["R2_test"]:.4f})', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'parity_plot.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"測試集 R2 = {r_step['R2_test']:.4f}, RMSE = {r_step['RMSE_test']:.4f}, MAE = {r_step['MAE_test']:.4f}")

測試集 R2 = 0.8703, RMSE = 2.9760, MAE = 2.3400


---
### 13. 交叉驗證穩定性評估

以 5 折交叉驗證評估逐步回歸精簡模型的泛化穩定性。

In [15]:
cv_scores = cross_val_score(
    LinearRegression(), X_train_scaled[step_feats], y_train, cv=5, scoring='r2')
print("逐步回歸精簡模型 5 折 CV R2:")
print(f"  各折: {np.round(cv_scores, 4)}")
print(f"  平均: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

# 與全特徵 OLS 比較
cv_full = cross_val_score(
    LinearRegression(), X_train_scaled, y_train, cv=5, scoring='r2')
print(f"\n全特徵 OLS 5 折 CV R2 平均: {cv_full.mean():.4f} (+/- {cv_full.std():.4f})")
print(f"精簡模型以 {len(step_feats)} 特徵達到與 {len(feature_names)} 特徵相當的性能。")

逐步回歸精簡模型 5 折 CV R2:
  各折: [0.8634 0.8532 0.8557 0.858  0.8433]
  平均: 0.8547 (+/- 0.0066)

全特徵 OLS 5 折 CV R2 平均: 0.8540 (+/- 0.0083)
精簡模型以 5 特徵達到與 8 特徵相當的性能。


---
### 14. 新操作條件預測

使用逐步回歸精簡模型預測 3 組新的操作條件下的產率。

In [16]:
new_conditions = pd.DataFrame({
    'Temperature_C': [90, 110, 100],
    'Pressure_bar': [2.0, 4.0, 3.0],
    'Catalyst_wt%': [1.0, 2.0, 1.5],
    'Time_min': [60, 150, 100],
    'Stirring_Speed_rpm': [300, 400, 250],
    'Feed_Rate_Lh': [20, 40, 30],
    'pH': [7.0, 6.5, 7.2],
    'Solvent_Ratio': [0.3, 0.2, 0.4],
})

new_scaled = pd.DataFrame(
    scaler.transform(new_conditions), columns=feature_names)
new_pred = m_step.predict(new_scaled[step_feats])

for i, p in enumerate(new_pred, 1):
    print(f"條件 {i}: 預測產率 = {p:.2f} %")

new_conditions_out = new_conditions.copy()
new_conditions_out['Predicted_Yield_%'] = new_pred.round(2)
print("\n", new_conditions_out.to_string(index=False))

條件 1: 預測產率 = 79.21 %
條件 2: 預測產率 = 104.75 %
條件 3: 預測產率 = 91.52 %

  Temperature_C  Pressure_bar  Catalyst_wt%  Time_min  Stirring_Speed_rpm  Feed_Rate_Lh  pH  Solvent_Ratio  Predicted_Yield_%
            90           2.0           1.0        60                 300            20 7.0            0.3              79.21
           110           4.0           2.0       150                 400            40 6.5            0.2             104.75
           100           3.0           1.5       100                 250            30 7.2            0.4              91.52


---
### 15. 模型持久化

儲存逐步回歸精簡模型、標準化器與中繼資料，供後續部署使用。

In [17]:
model_path = MODEL_DIR / 'stepwise_ols_model.pkl'
scaler_path = MODEL_DIR / 'stepwise_scaler.pkl'
meta_path = MODEL_DIR / 'stepwise_model_info.json'

joblib.dump(m_step, model_path)
joblib.dump(scaler, scaler_path)

model_info = {
    'model_type': 'Stepwise OLS (Forward Selection, AIC)',
    'selected_features': step_feats,
    'n_selected': len(step_feats),
    'all_features': feature_names,
    'R2_test': round(float(r_step['R2_test']), 4),
    'RMSE_test': round(float(r_step['RMSE_test']), 4),
    'MAE_test': round(float(r_step['MAE_test']), 4),
    'cv_r2_mean': round(float(cv_scores.mean()), 4),
    'cv_r2_std': round(float(cv_scores.std()), 4),
}
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(model_info, f, ensure_ascii=False, indent=2)

print("✓ 模型已儲存:")
print(f"  {model_path}")
print(f"  {scaler_path}")
print(f"  {meta_path}")

# 載入驗證
loaded = joblib.load(model_path)
check = loaded.predict(X_test_scaled[step_feats])
print(f"\n✓ 載入驗證通過, R2 = {r2_score(y_test, check):.4f}")

✓ 模型已儲存:
  D:\MyGit\ChemE-3590\Part_3\Unit10\outputs\P3_Unit10_Stepwise_Regression\models\stepwise_ols_model.pkl
  D:\MyGit\ChemE-3590\Part_3\Unit10\outputs\P3_Unit10_Stepwise_Regression\models\stepwise_scaler.pkl
  D:\MyGit\ChemE-3590\Part_3\Unit10\outputs\P3_Unit10_Stepwise_Regression\models\stepwise_model_info.json

✓ 載入驗證通過, R2 = 0.8703


---
### 16. 總結

本 Notebook 完整示範了逐步回歸的三種策略與實作：

1. **前向選擇 (AIC)**：從空模型逐步加入特徵，成功保留 4 個重要特徵。
2. **後向消除 (AIC)**：從全模型逐步移除冗餘特徵，結果與前向選擇一致。
3. **雙向逐步 (p-value)**：交替增刪，兼顧兩個方向的調整。
4. **sklearn SFS**：以交叉驗證 $R^2$ 為準則，與統計式逐步回歸互相對照。

**關鍵觀察**：
- 逐步回歸以少數特徵即可達到與全特徵 OLS 相當的測試集性能，且模型更精簡、可解釋性更高。
- 逐步回歸（Wrapper）與 Lasso（Embedded）通常選出相近的重要特徵，但原理不同。
- 逐步選擇存在推論偏誤與不穩定風險，正式統計推論時應謹慎，建議搭配交叉驗證驗證穩定性。

詳細理論與化工應用請參閱講義 [Unit10_Stepwise_Regression.md](Unit10_Stepwise_Regression.md)。